# KV-Cache Compression (Paper-Aligned Blueprints): PaLu, ThinKV, MiniCache, CommonKV, FullKV

This notebook applies compression on **actual KV cache tensors** and follows paper-aligned implementation patterns:

- **PaLu**: hidden-dimension compression (low-rank cached latent state + reconstruction)
- **ThinKV**: sequence/token eviction via attention-importance EMA
- **MiniCache**: cross-layer (depth) merge with direction/magnitude disentanglement
- **CommonKV**: cross-layer latent sharing + layer-specific residuals + adaptive budget
- **FullKV**: uncompressed baseline

> Note: exact production-quality speedups for ThinKV/PaLu/CommonKV need kernel-level integration (PagedAttention / fused reconstruction). This notebook implements the algorithmic logic in PyTorch for correctness-oriented experiments.

In [1]:
# !pip install -U torch transformers datasets accelerate sentencepiece pandas numpy tqdm

import json
import time
import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

In [2]:
from datasets import load_dataset, Dataset

# ds = load_dataset("simonjegou/ruler", "16384" , split="test")
# # print(ds[0])
# # ds.to_json("ruler_16384.jsonl")
# import os

# os.makedirs("ruler_data", exist_ok=True)

# for task_name in set(ds["task"]):
#     subset = ds.filter(lambda ex: ex["task"] == task_name)
#     out_path = f"ruler_data/{task_name}.jsonl"
#     subset.to_json(out_path)
#     print(f"Saved {task_name} → {out_path}")



## 1) Load model

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen1.5-0.5B" #"Qwen/Qwen2.5-7B-Instruct"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
    trust_remote_code=True,
)
if DEVICE == "cpu":
    model = model.to(DEVICE)
model.eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("loaded", MODEL_NAME)

`torch_dtype` is deprecated! Use `dtype` instead!
2026-03-22 01:58:36.763321: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-22 01:58:36.803493: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


loaded Qwen/Qwen1.5-0.5B


## 2) KV storage format and utilities

In [4]:
def _pair_layers(n_layers: int):
    pairs = []
    i = 0
    while i < n_layers:
        if i + 1 < n_layers:
            pairs.append((i, i + 1))
            i += 2
        else:
            pairs.append((i, None))
            i += 1
    return pairs


def _norm_dir(x, eps=1e-6):
    mag = torch.norm(x, dim=-1, keepdim=True)
    return x / (mag + eps), mag


def _bytes_tensor(x: torch.Tensor) -> int:
    return x.numel() * x.element_size()


def _stack_layers(layer_list):
    return tuple((k, v) for k, v in layer_list)

## 3) Compressor interfaces

In [5]:
class KVCompressor:
    name = "base"

    def reset(self):
        pass

    def initialize(self, past_key_values, layer_attn=None):
        """Initialize internal compressed state from model output."""
        raise NotImplementedError

    def update(self, past_key_values, layer_attn=None):
        """Update internal state after one new decode step."""
        raise NotImplementedError

    def reconstruct_for_model(self):
        """Return standard past_key_values tuple for next model forward."""
        raise NotImplementedError

    def cache_bytes(self):
        """Approximate bytes used by internal cache representation."""
        return 0

## 4) FullKV (baseline, no compression)

In [6]:
class FullKV(KVCompressor):
    name = "fullkv"

    def reset(self):
        self.past = None

    def initialize(self, past_key_values, layer_attn=None):
        self.past = tuple((k.detach(), v.detach()) for k, v in past_key_values)

    def update(self, past_key_values, layer_attn=None):
        self.past = tuple((k.detach(), v.detach()) for k, v in past_key_values)

    def reconstruct_for_model(self):
        return self.past

    def cache_bytes(self):
        if self.past is None:
            return 0
        return sum(_bytes_tensor(k) + _bytes_tensor(v) for k, v in self.past)

## 5) ThinKV (token-dimension, EMA attention + thought-group eviction)

In [7]:
class ThinKV(KVCompressor):
    name = "thinkv"

    def __init__(self, max_budget_tokens=2048, sink_tokens=32, local_tokens=512, ema_alpha=0.9, thought_block=32):
        self.max_budget_tokens = max_budget_tokens
        self.sink_tokens = sink_tokens
        self.local_tokens = local_tokens
        self.ema_alpha = ema_alpha
        self.thought_block = thought_block
        self.reset()

    def reset(self):
        self.past = None
        self.importance = None  # per-layer [B,H,T]

    def _update_importance(self, layer_attn):
        if layer_attn is None:
            return
        if self.importance is None:
            self.importance = [a.detach().clone() for a in layer_attn]
            return
        new_imp = []
        for old, a in zip(self.importance, layer_attn):
            Tn = a.shape[-1]
            To = old.shape[-1]
            if Tn > To:
                pad = torch.zeros(*old.shape[:-1], Tn - To, device=old.device, dtype=old.dtype)
                old = torch.cat([old, pad], dim=-1)
            old = old[..., :Tn]
            merged = self.ema_alpha * old + (1 - self.ema_alpha) * a
            new_imp.append(merged)
        self.importance = new_imp

    def _evict_indices(self, T, layer_imp, device):
        if T <= self.max_budget_tokens:
            return torch.arange(T, device=device)

        sink = torch.arange(0, min(self.sink_tokens, T), device=device)
        local_start = max(self.sink_tokens, T - self.local_tokens)
        local = torch.arange(local_start, T, device=device)

        protected = torch.cat([sink, local]).unique(sorted=True)
        available = max(0, self.max_budget_tokens - protected.numel())
        if available <= 0:
            return protected

        middle_start = sink.numel()
        middle_end = local_start
        if middle_end <= middle_start:
            return protected

        cand = torch.arange(middle_start, middle_end, device=device)
        # thought groups by fixed block (proxy for CoT segments)
        groups = (cand - middle_start) // self.thought_block
        imp = layer_imp.mean(dim=(0, 1)).index_select(0, cand)

        # aggregate per thought group
        unique_g = torch.unique(groups)
        group_scores = []
        for g in unique_g:
            mask = (groups == g)
            score = imp[mask].sum()
            group_scores.append((score, g))

        # keep highest-score groups until budget filled
        group_scores.sort(key=lambda x: float(x[0]), reverse=True)
        keep_tokens = []
        left = available
        for _, g in group_scores:
            tok = cand[groups == g]
            if tok.numel() <= left:
                keep_tokens.append(tok)
                left -= tok.numel()
            else:
                if left > 0:
                    keep_tokens.append(tok[:left])
                    left = 0
            if left <= 0:
                break

        kept_mid = torch.cat(keep_tokens) if keep_tokens else torch.empty(0, dtype=torch.long, device=device)
        return torch.cat([protected, kept_mid]).unique(sorted=True)

    def initialize(self, past_key_values, layer_attn=None):
        self.past = tuple((k.detach(), v.detach()) for k, v in past_key_values)
        self._update_importance(layer_attn)
        self._compress_now()

    def update(self, past_key_values, layer_attn=None):
        self.past = tuple((k.detach(), v.detach()) for k, v in past_key_values)
        self._update_importance(layer_attn)
        self._compress_now()

    def _compress_now(self):
        if self.past is None:
            return
        new_layers = []
        new_imp = []
        for li, (k, v) in enumerate(self.past):
            T = k.shape[2]
            imp = self.importance[li] if self.importance is not None else torch.ones(k.shape[0], k.shape[1], T, device=k.device)
            idx = self._evict_indices(T, imp, k.device)
            new_layers.append((k.index_select(2, idx), v.index_select(2, idx)))
            if self.importance is not None:
                new_imp.append(imp.index_select(-1, idx))
        self.past = tuple(new_layers)
        if self.importance is not None:
            self.importance = new_imp

    def reconstruct_for_model(self):
        return self.past

    def cache_bytes(self):
        if self.past is None:
            return 0
        val = sum(_bytes_tensor(k) + _bytes_tensor(v) for k, v in self.past)
        if self.importance is not None:
            val += sum(_bytes_tensor(x) for x in self.importance)
        return val

## 6) MiniCache (depth-dimension, adjacent-layer direction/magnitude merge)

In [8]:
class MiniCache(KVCompressor):
    name = "minicache"

    def __init__(self, angle_threshold=0.95):
        self.angle_threshold = angle_threshold
        self.reset()

    def reset(self):
        self.n_layers = 0
        self.pairs = []
        self.shared = {}  # (l,l1)-> dict with shared dir/mag/mask/original_unmerged

    def _compress_pair(self, K0, V0, K1, V1):
        # K/V: [B,H,T,D]
        dir0, mag0 = _norm_dir(K0)
        dir1, mag1 = _norm_dir(K1)
        cos = (dir0 * dir1).sum(dim=-1, keepdim=True)
        shared_dir = F.normalize(dir0 + dir1, dim=-1)
        mask = (cos > self.angle_threshold)

        final_dir = torch.where(mask, shared_dir, dir0)
        data = {
            "k_dir": final_dir.detach(),
            "k_mag0": mag0.detach(),
            "k_mag1": mag1.detach(),
            "k_mask": mask.detach(),
            "k_unmerged1": K1.detach(),
            "v0": V0.detach(),
            "v1": V1.detach(),
        }
        return data

    def _reconstruct_pair(self, data):
        K0 = data["k_dir"] * data["k_mag0"]
        K1_merge = data["k_dir"] * data["k_mag1"]
        K1 = torch.where(data["k_mask"], K1_merge, data["k_unmerged1"])
        return K0, data["v0"], K1, data["v1"]

    def initialize(self, past_key_values, layer_attn=None):
        self.n_layers = len(past_key_values)
        self.pairs = _pair_layers(self.n_layers)
        self.shared = {}
        for l0, l1 in self.pairs:
            if l1 is None:
                self.shared[(l0, l1)] = {"single": (past_key_values[l0][0].detach(), past_key_values[l0][1].detach())}
            else:
                K0, V0 = past_key_values[l0]
                K1, V1 = past_key_values[l1]
                self.shared[(l0, l1)] = self._compress_pair(K0, V0, K1, V1)

    def update(self, past_key_values, layer_attn=None):
        self.initialize(past_key_values, layer_attn)

    def reconstruct_for_model(self):
        layers = [None] * self.n_layers
        for (l0, l1), data in self.shared.items():
            if l1 is None:
                layers[l0] = data["single"]
            else:
                K0, V0, K1, V1 = self._reconstruct_pair(data)
                layers[l0] = (K0, V0)
                layers[l1] = (K1, V1)
        return _stack_layers(layers)

    def cache_bytes(self):
        total = 0
        for (_, _), d in self.shared.items():
            for v in d.values():
                if torch.is_tensor(v):
                    total += _bytes_tensor(v)
                elif isinstance(v, tuple):
                    total += sum(_bytes_tensor(x) for x in v)
        return total

## 7) CommonKV (depth-dimension, shared latent cache + residual adapters + adaptive budget)

In [9]:
class CommonKV(KVCompressor):
    name = "commonkv"

    def __init__(self, base_shared_ratio=0.75, min_shared_ratio=0.40, max_shared_ratio=0.95):
        self.base_shared_ratio = base_shared_ratio
        self.min_shared_ratio = min_shared_ratio
        self.max_shared_ratio = max_shared_ratio
        self.reset()

    def reset(self):
        self.n_layers = 0
        self.pairs = []
        self.data = {}

    def _adaptive_ratio(self, X0, X1):
        sim = F.cosine_similarity(X0.flatten(2), X1.flatten(2), dim=-1).mean()
        sim_f = float(torch.clamp(sim, -1, 1))
        ratio = self.base_shared_ratio + 0.2 * sim_f
        ratio = max(self.min_shared_ratio, min(self.max_shared_ratio, ratio))
        return ratio

    def _compress_pair(self, K0, V0, K1, V1):
        # shared latent + residuals
        ratio = self._adaptive_ratio(K0, K1)
        shared_k = 0.5 * (K0 + K1)
        shared_v = 0.5 * (V0 + V1)
        dK0 = K0 - shared_k
        dK1 = K1 - shared_k
        dV0 = V0 - shared_v
        dV1 = V1 - shared_v

        # adaptive residual sparsification budget
        keep = max(1, int(shared_k.shape[2] * ratio))
        if keep < shared_k.shape[2]:
            idx = torch.linspace(0, shared_k.shape[2]-1, keep, device=shared_k.device).long()
            shared_k = shared_k.index_select(2, idx)
            shared_v = shared_v.index_select(2, idx)
            dK0 = dK0.index_select(2, idx)
            dK1 = dK1.index_select(2, idx)
            dV0 = dV0.index_select(2, idx)
            dV1 = dV1.index_select(2, idx)

        return {
            "shared_k": shared_k.detach(),
            "shared_v": shared_v.detach(),
            "dK0": dK0.detach(),
            "dK1": dK1.detach(),
            "dV0": dV0.detach(),
            "dV1": dV1.detach(),
        }

    def initialize(self, past_key_values, layer_attn=None):
        self.n_layers = len(past_key_values)
        self.pairs = _pair_layers(self.n_layers)
        self.data = {}
        for l0, l1 in self.pairs:
            if l1 is None:
                self.data[(l0, l1)] = {"single": (past_key_values[l0][0].detach(), past_key_values[l0][1].detach())}
            else:
                K0, V0 = past_key_values[l0]
                K1, V1 = past_key_values[l1]
                self.data[(l0, l1)] = self._compress_pair(K0, V0, K1, V1)

    def update(self, past_key_values, layer_attn=None):
        self.initialize(past_key_values, layer_attn)

    def reconstruct_for_model(self):
        layers = [None] * self.n_layers
        for (l0, l1), d in self.data.items():
            if l1 is None:
                layers[l0] = d["single"]
            else:
                K0 = d["shared_k"] + d["dK0"]
                K1 = d["shared_k"] + d["dK1"]
                V0 = d["shared_v"] + d["dV0"]
                V1 = d["shared_v"] + d["dV1"]
                layers[l0] = (K0, V0)
                layers[l1] = (K1, V1)
        return _stack_layers(layers)

    def cache_bytes(self):
        total = 0
        for (_, _), d in self.data.items():
            for v in d.values():
                if torch.is_tensor(v):
                    total += _bytes_tensor(v)
                elif isinstance(v, tuple):
                    total += sum(_bytes_tensor(x) for x in v)
        return total

## 8) PaLu (hidden-dimension low-rank cache with on-the-fly reconstruction)

In [10]:
class PaLu(KVCompressor):
    name = "palu"

    def __init__(self, rank=64):
        self.rank = rank
        self.reset()

    def reset(self):
        self.lowrank = []  # per layer: tuples of z_k,z_v,b_k,b_v
        self.n_layers = 0

    def _factorize(self, X, r):
        # X: [B,H,T,D], compress hidden dim D -> r, keep token dimension T
        B, H, T, D = X.shape
        X2 = X.reshape(B*H*T, D)
        m = X2.mean(0, keepdim=True)
        Xc = X2 - m
        try:
            U, S, Vh = torch.linalg.svd(Xc, full_matrices=False)
            rr = min(r, Vh.shape[0])
            Bm = Vh[:rr, :]            # [r,D]
            Z = X2 @ Bm.t()            # [BHT,r]
            return Z.reshape(B, H, T, rr), Bm, m
        except Exception:
            # fallback no compression
            eye = torch.eye(D, device=X.device, dtype=X.dtype)
            return X, eye, torch.zeros_like(m)

    def _reconstruct(self, Z, Bm, mean, D):
        if Bm.shape[0] == D and torch.allclose(Bm, torch.eye(D, device=Bm.device, dtype=Bm.dtype)):
            return Z
        B, H, T, R = Z.shape
        Z2 = Z.reshape(B*H*T, R)
        X2 = Z2 @ Bm + mean
        return X2.reshape(B, H, T, D)

    def initialize(self, past_key_values, layer_attn=None):
        self.n_layers = len(past_key_values)
        self.lowrank = []
        for k, v in past_key_values:
            Dk = k.shape[-1]
            Dv = v.shape[-1]
            zk, bk, mk = self._factorize(k.detach(), self.rank)
            zv, bv, mv = self._factorize(v.detach(), self.rank)
            self.lowrank.append((zk, zv, bk, bv, mk, mv, Dk, Dv))

    def update(self, past_key_values, layer_attn=None):
        self.initialize(past_key_values, layer_attn)

    def reconstruct_for_model(self):
        layers = []
        for (zk, zv, bk, bv, mk, mv, Dk, Dv) in self.lowrank:
            k = self._reconstruct(zk, bk, mk, Dk)
            v = self._reconstruct(zv, bv, mv, Dv)
            layers.append((k, v))
        return _stack_layers(layers)

    def cache_bytes(self):
        total = 0
        for (zk, zv, bk, bv, mk, mv, _, _) in self.lowrank:
            for t in [zk, zv, bk, bv, mk, mv]:
                total += _bytes_tensor(t)
        return total

## 9) One generation loop that uses compressor internal state

In [11]:
@torch.no_grad()
def generate_with_compressor(prompt: str, compressor: KVCompressor, max_new_tokens=128, temperature=0.0):
    compressor.reset()
    inp = tokenizer(prompt, return_tensors="pt")
    inp = {k: v.to(model.device) for k, v in inp.items()}

    # Prefill
    out = model(**inp, use_cache=True, output_attentions=True)
    layer_attn = [a[:, :, -1, :].detach() for a in out.attentions] if out.attentions is not None else None
    compressor.initialize(out.past_key_values, layer_attn=layer_attn)

    logits = out.logits[:, -1, :]
    gen_ids = []

    for _ in range(max_new_tokens):
        if temperature > 0:
            probs = torch.softmax(logits / temperature, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
        else:
            next_tok = torch.argmax(logits, dim=-1, keepdim=True)

        tok = next_tok.item()
        if tok == tokenizer.eos_token_id:
            break
        gen_ids.append(tok)

        past_for_model = compressor.reconstruct_for_model()
        out = model(
            input_ids=next_tok,
            past_key_values=past_for_model,
            use_cache=True,
            output_attentions=True,
        )
        layer_attn = [a[:, :, -1, :].detach() for a in out.attentions] if out.attentions is not None else None
        compressor.update(out.past_key_values, layer_attn=layer_attn)
        logits = out.logits[:, -1, :]

    return tokenizer.decode(gen_ids, skip_special_tokens=True), compressor.cache_bytes()

## 10) Bench data + metrics (LongBench + Ruler in one pipeline)

In [12]:
TOKEN_RE = re.compile(r"\w+", re.UNICODE)

def _words(s):
    return TOKEN_RE.findall((s or "").lower())

def em(pred, refs):
    p = (pred or "").strip().lower()
    return float(any(p == (r or "").strip().lower() for r in refs))

def f1(pred, refs):
    p = _words(pred)
    if not p: return 0.0
    best=0.0
    for r in refs:
        t=_words(r)
        if not t: continue
        c=0
        cnt={}
        for x in t: cnt[x]=cnt.get(x,0)+1
        for x in p:
            if cnt.get(x,0)>0:
                c+=1; cnt[x]-=1
        if c==0: continue
        pr=c/len(p); rc=c/len(t)
        best=max(best,2*pr*rc/(pr+rc))
    return best

def score(pred, refs, metric):
    m=metric.lower()
    if m in {"em","exact_match"}: return em(pred, refs)
    return f1(pred, refs)

@dataclass
class Ex:
    eid: str
    benchmark: str
    group: str
    task: str
    prompt: str
    refs: List[str]
    metric: str


def load_longbench(task_specs, split="test", limit=20):
    from datasets import load_dataset
    rows=[]
    for s in task_specs:
        ds=load_dataset("THUDM/LongBench", s["task"], split=split)
        if limit: ds=ds.select(range(min(limit,len(ds))))
        for i,ex in enumerate(ds):
            prompt=ex.get("prompt") or ex.get("input") or ""
            if not prompt and "context" in ex and "question" in ex:
                prompt="Context:\n{}\n\nQuestion: {}\nAnswer:".format(ex['context'], ex['question'])
            refs=ex.get("answers") or ex.get("answer") or ex.get("output") or [""]
            if not isinstance(refs,list): refs=[str(refs)]
            rows.append(Ex(f"lb::{s['task']}::{i}","LongBench",s["group"],s["task"],str(prompt),[str(x) for x in refs],s.get("metric","f1")))
    return rows


def load_ruler(root_dir, task_specs, limit=20):
    rows=[]
    root=Path(root_dir)
    for s in task_specs:
        fp=root/f"{s['task']}.jsonl"
        if not fp.exists():
            print("missing", fp)
            continue
        with fp.open("r", encoding="utf-8") as f:
            for i,line in enumerate(f):
                if limit and i>=limit: break
                ex=json.loads(line)
                prompt=ex.get("prompt") or ex.get("input") or ""
                refs=ex.get("answers") or ex.get("answer") or ex.get("output") or [""]
                if not isinstance(refs,list): refs=[str(refs)]
                rows.append(Ex(f"ru::{s['task']}::{i}","Ruler",s["group"],s["task"],str(prompt),[str(x) for x in refs],s.get("metric","em")))
    return rows

## 11) Unified eval + one final table

In [14]:
COMPRESSORS = {
    "fullkv": FullKV(),
    "thinkv": ThinKV(max_budget_tokens=2048, sink_tokens=32, local_tokens=512),
    "minicache": MiniCache(angle_threshold=0.95),
    "commonkv": CommonKV(base_shared_ratio=0.75),
    "palu": PaLu(rank=64),
}

LONGBENCH_TASKS = [
    {"task":"narrativeqa","group":"QA","metric":"f1"},
    {"task":"gov_report","group":"Sum.","metric":"f1"},
    {"task":"trec","group":"FShot","metric":"em"},
    {"task":"hotpotqa","group":"Synth.","metric":"f1"},
    {"task":"lcc","group":"Code","metric":"em"},
]
# RULER_TASKS = [
#     {"task":"ns","group":"NS","metric":"em"},
#     {"task":"nmk","group":"NMK","metric":"em"},
#     {"task":"nmq","group":"NMQ","metric":"em"},
#     {"task":"nmv","group":"NMV","metric":"em"},
#     {"task":"qa","group":"QA","metric":"f1"},
#     {"task":"vt","group":"VT","metric":"em"},
#     {"task":"fwe","group":"FWE","metric":"em"},
# ]

RULER_TASKS = [
    # group labels: NS, NMK, NMQ, NMV, QA, VT, FWE, CWE
    {"task": "cwe", "group": "CWE", "metric": "em"},
    {"task": "niah_multikey_1", "group": "NMK", "metric": "em"},
    {"task": "niah_multikey_2", "group": "NMK", "metric": "em"},
    {"task": "niah_multikey_3", "group": "NMK", "metric": "em"},
    {"task": "niah_multiquery", "group": "NMQ", "metric": "em"},
    {"task": "niah_multivalue", "group": "NMV", "metric": "em"},
    {"task": "niah_single_1", "group": "NS", "metric": "em"},
    {"task": "niah_single_2", "group": "NS", "metric": "em"},
    {"task": "niah_single_3", "group": "NS", "metric": "em"},
    {"task": "qa_1", "group": "QA", "metric": "f1"},
    {"task": "qa_2", "group": "QA", "metric": "f1"},
    {"task": "vt", "group": "VT", "metric": "em"},
    {"task": "fwe", "group": "FWE", "metric": "em"},
]

examples=[]
examples += load_longbench(LONGBENCH_TASKS, split="test", limit=10)
examples += load_ruler("./ruler_data", RULER_TASKS, limit=10)
print("examples", len(examples))

rows=[]
for ex in tqdm(examples, desc="eval"):
    for name, comp in COMPRESSORS.items():
        t0=time.perf_counter()
        err=""
        try:
            pred, cbytes = generate_with_compressor(ex.prompt, comp, max_new_tokens=128, temperature=0.0)
        except Exception as e:
            pred=""; cbytes=0; err=str(e)
        dt=time.perf_counter()-t0
        sc=0.0 if err else score(pred, ex.refs, ex.metric)
        rows.append({
            "eid":ex.eid,"benchmark":ex.benchmark,"group":ex.group,"task":ex.task,
            "method":name,"metric":ex.metric,"score":sc,"latency_s":dt,
            "cache_bytes":cbytes,"error":err,
        })

results_df = pd.DataFrame(rows)

summary = results_df.pivot_table(index="method", columns=["benchmark","group"], values="score", aggfunc="mean")
summary.columns=[f"{b}:{g}" for b,g in summary.columns]
summary["Avg."] = results_df.groupby("method")["score"].mean()
summary["Latency(s)"] = results_df.groupby("method")["latency_s"].mean()
summary["CacheMB"] = results_df.groupby("method")["cache_bytes"].mean()/(1024**2)
summary = summary.sort_values("Avg.", ascending=False)
summary

examples 180


eval:   0%|          | 0/180 [00:00<?, ?it/s]

/home/seqaeon/Downloads/venv/lib/python3.12/site-packages/transformers/utils/generic.py:1014: UserWarning: `output_attentions=True` is not supported with `attn_implementation` other than ['eager', 'eager_paged', 'flex_attention']. Please use `model.set_attn_implementation('eager')` to enable capturing attention outputs.
  warnings.warn(
`sdpa` attention does not support `output_attentions=True` or `head_mask`. Please set your attention to `eager` if you want any of these features.


,LongBench:Code,LongBench:FShot,LongBench:QA,LongBench:Sum.,LongBench:Synth.,Ruler:CWE,Ruler:FWE,Ruler:NMK,Ruler:NMQ,Ruler:NMV,Ruler:NS,Ruler:QA,Ruler:VT,Avg.,Latency(s),CacheMB
method,,,,,,,,,,,,,,,,
commonkv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.009186,0.0
fullkv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.010093,0.0
minicache,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.007695,0.0
palu,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.012370,0.0
thinkv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.009592,0.0


In [ ]:
out=Path("artifacts")
out.mkdir(exist_ok=True)
results_df.to_csv(out/"kvcache_eval_raw.csv", index=False)
summary.to_csv(out/"kvcache_eval_summary.csv")
print("saved artifacts/kvcache_eval_raw.csv and artifacts/kvcache_eval_summary.csv")